# Feature Engineering — Project Outcome Prediction

Joins each engagement with client + lead-consultant attributes and derives
pre-project features. EXCLUDES post-project leakage (actual_spend_gbp, margin_pct, status).

**Reads:** `silver_engagements`, `silver_clients`, `silver_consultants`  **Writes:** `gold_ml_features`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, when

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
eng = spark.read.table('silver_engagements')
cl = spark.read.table('silver_clients')
cn = spark.read.table('silver_consultants')
print(f'engagements={eng.count():,} clients={cl.count():,} consultants={cn.count():,}')

required = {'engagement_id', 'client_id', 'lead_consultant_id', 'is_over_budget', 'budget_gbp'}
missing = required - set(eng.columns)
if missing:
    raise ValueError(f'silver_engagements missing columns {sorted(missing)}. Regenerate data and rerun bronze/silver.')

In [ ]:
# Lead consultant attrs (rename to avoid clash with engagement.practice / client.region).
lead = cn.select(
    col('consultant_id').alias('lead_consultant_id'),
    col('grade').alias('lead_grade'),
    col('years_experience').alias('lead_experience'),
    col('daily_rate_gbp').alias('lead_daily_rate'),
)

# Join attributes + select pre-project features. EXCLUDE leakage (actual_spend_gbp, margin_pct, status).
ml_features = (
    eng.select(
        'engagement_id', 'client_id', 'lead_consultant_id', 'practice',
        'budget_gbp', 'headcount', 'planned_duration_days',
        col('is_over_budget').alias('had_overrun'),
    )
    .join(cl.select('client_id', 'industry', 'tier', 'region',
                    'contract_value_gbp', 'relationship_years', 'nps_score'),
          'client_id', 'left')
    .join(lead, 'lead_consultant_id', 'left')
    .na.fill(0)
    .na.fill('unknown', subset=['practice', 'industry', 'tier', 'region', 'lead_grade'])
    .withColumn('feature_timestamp', current_timestamp())
)

total_rows = ml_features.count()
positive_rows = ml_features.filter(col('had_overrun') == 1).count()
overrun_rate = (positive_rows / total_rows * 100) if total_rows else 0.0

if total_rows < 1000 or positive_rows < max(10, int(total_rows * 0.01)):
    raise ValueError(
        f'Label quality check failed: only {positive_rows}/{total_rows} overrun rows '
        f'({overrun_rate:.2f}%). Check is_over_budget typing and source data.'
    )

ml_features.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_features')
print(f'Gold ML features written: {total_rows:,} rows | overrun rate {overrun_rate:.1f}%')

In [ ]:
spark.sql('OPTIMIZE gold_ml_features')
print('Feature table optimized')